# Assignment 2: Milestone I Natural Language Processing
## Task 2&3
#### Student Name: Shiv Mahesh Bhosale
#### Student ID: S4192953


Environment: Python 397 and Jupyter notebook

Libraries used: 
- pandas
- numpy
- regex
- sklearn
- gensim

## Introduction
In this task, multiple feature feature representations are generated from the processed clothing reviews.

The representations include:
- Bag-of-words
- Unweighted word embedding vectors
- TF-IDF weighted embedding vectors

These representations will later be used for classification experiments.

## Importing libraries 

In [1]:
import pandas as pd
import numpy as np
import regex as re
from collections import Counter
from itertools import chain
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score
import gensim
import gensim.downloader as api
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import cross_val_score

The commands which are commented below are used to install gensim and solve the errors while running the notebook in py397 version.

In [2]:
# !pip install pandas numpy scikit-learn gensim nltk

In [3]:
# import sys

# print(sys.executable)
# print(sys.version)

In [4]:
# import sys
# !{sys.executable} -m pip install pandas numpy scikit-learn gensim nltk

## Task 2. Generating Feature Representations for Clothing Items Reviews

In this task, multiple feature representations are generated from the processed clothing reviews.

The representations include:
- Bag-of-Words count vectors
- Unweighted word embedding vectors
- TF-IDF weighted embedding vectors

Loading the 'vocab.txt' file and creating a vocab for further tasks.

In [5]:
def load_vocab(filepath='vocab'):
    vocab = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                word, idx = line.rsplit(':', 1)
                vocab[word] = int(idx)
    return vocab

In [6]:
vocab = load_vocab('vocab.txt')
index_to_word = {idx: word for word, idx in vocab.items()}

In [7]:
print(f"Vocabulary size: {len(vocab)}")
print(f"\nFirst 10 vocab entries:")
for word, idx in list(vocab.items())[:10]:
    print(f"  {word}:{idx}")

Vocabulary size: 7529

First 10 vocab entries:
  a-cup:0
  a-flutter:1
  a-frame:2
  a-kind:3
  a-line:4
  a-lines:5
  a-symmetric:6
  aa:7
  ab:8
  abbey:9


Loading the 'processed.csv' file and creating a dataframe df.

In [8]:
df = pd.read_csv('./processed.csv')

In [9]:
df

,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,1077,60,Some major design flaws,high hopes wanted work initially petite usual ...,3,0,0,General,Dresses,Dresses
1,1049,50,My favorite buy!,jumpsuit fun flirty fabulous time compliments,5,1,0,General Petite,Bottoms,Pants
2,847,47,Flattering shirt,shirt due adjustable front tie length leggings...,5,1,6,General,Tops,Blouses
3,1080,49,Not for the very petite,tracy reese dresses petite feet tall brand pre...,2,0,4,General,Dresses,Dresses
4,858,39,Cagrcoal shimmer fun,basket hte person store pick teh pale hte gorg...,5,1,1,General Petite,Tops,Knits
...,...,...,...,...,...,...,...,...,...,...
19657,1104,34,Great dress for many occasions,happy snag price easy slip cut combo,5,1,0,General Petite,Dresses,Dresses
19658,862,48,Wish it was made of cotton,reminds maternity clothes stretchy shiny mater...,3,1,0,General Petite,Tops,Knits
19659,1104,31,"Cute, but see through",worked glad store order online,3,0,1,General Petite,Dresses,Dresses
19660,1084,28,"Very cute dress, perfect for summer parties an...",wedding summer medium waist perfectly long big...,3,1,2,General,Dresses,Dresses


In [1]:
df["Review Text"].isnull().sum()

NameError: name 'df' is not defined

In [11]:
df['Review Text'] = df['Review Text'].fillna('')

In [12]:
tk_reviews = [review.split(' ') for review in df['Review Text'].tolist()]

In [13]:
tk_reviews[1]

['jumpsuit', 'fun', 'flirty', 'fabulous', 'time', 'compliments']

In [14]:
joined_reviews = [' '.join(tokens) for tokens in tk_reviews]

In [15]:
joined_reviews[1]

'jumpsuit fun flirty fabulous time compliments'

In [16]:
len(tk_reviews)

19662

### Bag-of-words Count Vector Representation

A bag-of-word count vector is created for each review.


In [17]:
cVectorizer = CountVectorizer(analyzer="word", vocabulary=vocab)
count_features = cVectorizer.fit_transform(joined_reviews)

In [18]:
count_features.shape

(19662, 7529)

### Saving Count Vector Outputs

The generated sparse count vectors are saved into `count_vectors.txt` using the required assignment format.

In [19]:
out_file = open('count_vectors.txt', 'w')

In [20]:
for doc_id in range(count_features.shape[0]):
    row = count_features[doc_id].tocoo()
    pairs = ','.join(
        f"{col}:{val}"
        for col, val in sorted(zip(row.col, row.data))
    )
    out_file.write(f"#{doc_id},{pairs}\n")
out_file.close()

In [21]:
with open('count_vectors.txt') as f:
    lines = f.read().splitlines()

In [22]:
len(lines)

19662

In [23]:
lines[0]

'#0,687:1,1028:1,1716:1,1792:1,2289:1,2481:1,2602:1,2892:2,3010:1,3087:1,3193:1,3258:1,3549:2,3552:1,3832:1,3934:1,4224:2,4234:1,4427:1,4639:2,5260:1,5668:1,6726:1,7092:1,7207:1,7406:1,7520:1,7522:1'

### TF-IDF Vectors

In [24]:
tVectorizer = TfidfVectorizer(analyzer="word", vocabulary=vocab)
tfidf_features = tVectorizer.fit_transform(joined_reviews)

In [25]:
tfidf_features.shape

(19662, 7529)

In [26]:
sample = tfidf_features[0].tocoo()
top = sorted(zip(sample.col, sample.data), key=lambda x: -x[1])[:8]

print("Top TF-IDF words in review 0:")
for index, score in top:
    print(f"  '{index_to_word[int(index)]}': {score:.4f}")

Top TF-IDF words in review 0:
  'net': 0.4868
  'half': 0.2999
  'layer': 0.2739
  'outrageously': 0.2568
  'directly': 0.2204
  'reordered': 0.2012
  'major': 0.1947
  'imo': 0.1931


### Word Embedding Model

GloVe word embeddings are used to generate dense semantic vector representations for reviews.

In [27]:
embedding_model = api.load('glove-wiki-gigaword-100')

In [28]:
embedding_dim = embedding_model.vector_size
print('Embedding dimension:', embedding_dim)

Embedding dimension: 100


In [29]:
def getDocVec_unweighted(review, model, vec_size):
    vectors = []
    for w in review:
        if w in model:
            vectors.append(model[w])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vec_size)

In [30]:
unweighted_doc_vectors = np.array(
    [getDocVec_unweighted(review, embedding_model, embedding_dim) 
     for review in tk_reviews]
)
unweighted_doc_vectors.shape

(19662, 100)

In [31]:
unweighted_doc_vectors[0]

array([-2.37603962e-01,  1.87611982e-01,  2.06782848e-01, -1.72068402e-01,
        6.61378056e-02,  1.36691332e-02, -2.33567227e-02,  1.40242264e-01,
       -2.55546463e-03,  2.42306277e-01,  1.41616151e-01,  1.25330091e-02,
        1.93498850e-01,  1.08647849e-02,  1.96627885e-01, -4.82560620e-02,
        4.99383407e-03,  3.68231684e-02,  7.15183616e-02, -1.06607646e-01,
        1.03885695e-01,  5.64851873e-02,  1.00387305e-01, -3.86443082e-03,
        2.11305186e-01, -1.03310585e-01, -1.18373983e-01, -3.17010999e-01,
       -2.63772845e-01, -2.59434611e-01,  3.39593329e-02,  2.22990528e-01,
       -8.36506262e-02, -2.30186611e-01,  1.45958006e-01,  1.70477778e-01,
        1.00440197e-02,  6.75151944e-02,  1.63908079e-02, -8.94875526e-02,
        8.85799080e-02, -4.08290982e-01, -3.43823247e-02,  2.66765319e-02,
       -4.74542156e-02,  7.01193810e-02,  1.43701673e-01,  3.43122631e-02,
        2.44205575e-02, -2.67926276e-01, -9.67212580e-03, -1.35283088e-02,
        1.20197330e-03,  

### TF-IDF Weighted Embeddings

In [32]:
tfidf_vectorizer = TfidfVectorizer(
    analyzer="word",
    vocabulary=vocab
)

tfidf_features = tfidf_vectorizer.fit_transform(joined_reviews)

print("TF-IDF matrix shape:", tfidf_features.shape)

TF-IDF matrix shape: (19662, 7529)


In [33]:
def getDocVec_weighted(row_index, review, model, tfidf_matrix, vocab, embedding_dim):
   
    words = str(review).split()

    weighted_vector = np.zeros(embedding_dim)
    total_weight    = 0

    for word in words:
        if word in model and word in vocab:
            vocab_index = vocab[word]                           # get word index from vocab dict
            weight      = tfidf_matrix[row_index, vocab_index] # TF-IDF score for this word in this doc

            weighted_vector += model[word] * weight
            total_weight    += weight

    if total_weight == 0:
        return np.zeros(embedding_dim)

    return weighted_vector / total_weight

In [34]:
weighted_doc_vectors = np.array([
    getDocVec_weighted(i, review, embedding_model, tfidf_features, vocab, embedding_dim)
    for i, review in enumerate(df['Review Text'])
])

print("Weighted vectors shape:", weighted_doc_vectors.shape)

Weighted vectors shape: (19662, 100)


In [35]:
weighted_doc_vectors[0]

array([-0.17441661,  0.18540395,  0.30812363, -0.19346222,  0.12216701,
        0.05211715,  0.0314983 ,  0.10174598,  0.00406353,  0.16109891,
        0.21541216,  0.08326695,  0.13471637, -0.01027021,  0.28298694,
       -0.06707228, -0.08769617,  0.05520173,  0.11328442, -0.06240552,
        0.18406146,  0.0959304 ,  0.05547843,  0.24010124,  0.20323547,
       -0.13981546, -0.04189288, -0.22727854, -0.25998257, -0.39795513,
        0.06823332,  0.28449805, -0.08575211, -0.25840537,  0.15151444,
        0.11191569, -0.05557328,  0.1522054 , -0.04125074, -0.06995603,
        0.14335732, -0.42803139, -0.06907106,  0.11697324, -0.00592992,
        0.0994627 ,  0.0824857 , -0.0068793 , -0.03920164, -0.27981272,
       -0.02275862,  0.0276556 , -0.01625947,  0.63171904, -0.34020019,
       -1.41825161, -0.05474229, -0.16502886,  1.1212131 ,  0.32256497,
        0.06640504,  0.33642742, -0.34816536,  0.31385802,  0.29781783,
       -0.14635509,  0.24605817,  0.02502787,  0.28383756, -0.21

In [36]:
print("Unweighted (review 0):", unweighted_doc_vectors[0][:8])
print("Weighted   (review 0):", weighted_doc_vectors[0][:8])

Unweighted (review 0): [-0.23760396  0.18761198  0.20678285 -0.1720684   0.06613781  0.01366913
 -0.02335672  0.14024226]
Weighted   (review 0): [-0.17441661  0.18540395  0.30812363 -0.19346222  0.12216701  0.05211715
  0.0314983   0.10174598]


### Saving Embedding Representations
The generated embedding vectors are saved for future reuse and experimentation.

In [37]:
np.savetxt("unweighted_vectors.txt", unweighted_doc_vectors)
np.savetxt("weighted_vectors.txt", weighted_doc_vectors)

## Task 3. Clothing Review Classification

In this task, Logistic Regression is used as the selected machine learning model to classify whether a clothing item review is recommended or not. Two experiments are conducted. The first experiment compares the feature representations generated in Task 2. The second experiment checks whether using extra text information, such as review title, improves classification accuracy. A 5-fold cross validation is used for evaluation.

### preparing target variable
The target variable for classification is the Recommended IND column. This column shows whether the clothing item was recommended (1) or not (0).

df.columns.tolist()

In [38]:
y = df["Recommended IND"]

The target variable is checked before modelling because the classes may not be balanced. If one class appears much more than the other, accuracy alone may not fully explain the model performance. Therefore, precision, recall, and F1-score are also used later with accuracy.

In [39]:
print(y.value_counts())

Recommended IND
1    16087
0     3575
Name: count, dtype: int64


### Machine learning model

Logistic Regression is used as the classification model. Only one machine learning model is selected, following the tutor's instruction. The same model is used for all feature representations so the comparison is fair.

In [40]:
model = LogisticRegression(max_iter=1000)
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [41]:
scoring_metrics = {
    "accuracy" : "accuracy",
    "precision": "precision",
    "recall"   : "recall",
    "f1"       : "f1"
}

## Q1: Language model comparison

This method compares the three feature representations generated in Task 2: count vectors, unweighted GloVe vectors, and TF-IDF weighted GloVe vectors. The same Logistic Regression model and 5-fold cross validation are used for all three representations.

#### creating count vectors

In [42]:
from collections import Counter

count_vectors = []

for review in tk_reviews:
    freq = Counter(review)
    sparse_vector = [(vocab[word], count) 
                     for word, count in freq.items() 
                     if word in vocab]
    count_vectors.append(sparse_vector)


The count_vectors list is converted into a sparse matrix for use with sklearn.

In [43]:
row_indices = []
col_indices = []
data_values = []

for row_id, sparse_vector in enumerate(count_vectors):
    for word_index, word_freq in sparse_vector:
        row_indices.append(row_id)
        col_indices.append(word_index)
        data_values.append(word_freq)

X_count = csr_matrix(
    (data_values, (row_indices, col_indices)),
    shape=(len(count_vectors), len(vocab))
)

print("Count vector matrix shape:", X_count.shape)

Count vector matrix shape: (19662, 7529)


In [44]:
print("Count vector matrix shape:", X_count.shape)

Count vector matrix shape: (19662, 7529)


#### To Evaluate count vector representation using 5-fold cross validation

In [45]:
count_results = cross_validate(
    model,
    X_count,
    y,
    cv=cv,
    scoring=scoring_metrics
)

print("Count vector accuracy scores:", count_results["test_accuracy"])
print("Mean accuracy:"  , count_results["test_accuracy"].mean())
print("Mean precision:" , count_results["test_precision"].mean())
print("Mean recall:"    , count_results["test_recall"].mean())
print("Mean F1-score:"  , count_results["test_f1"].mean())

Count vector accuracy scores: [0.87414188 0.87541317 0.87716175 0.87894201 0.87614446]
Mean accuracy: 0.8763606533546776
Mean precision: 0.9040728773181547
Mean recall: 0.9496488801625453
Mean F1-score: 0.9262999978522967


### Unweighted embedding features

#### To Evaluate unweighted embedding representation using 5-fold cross validation

In [46]:
unweighted_results = cross_validate(
    model,
    unweighted_doc_vectors,
    y,
    cv=cv,
    scoring=scoring_metrics
)

print("Unweighted embedding accuracy scores:", unweighted_results["test_accuracy"])
print("Mean accuracy:"  , unweighted_results["test_accuracy"].mean())
print("Mean precision:" , unweighted_results["test_precision"].mean())
print("Mean recall:"    , unweighted_results["test_recall"].mean())
print("Mean F1-score:"  , unweighted_results["test_f1"].mean())

Unweighted embedding accuracy scores: [0.84388508 0.84261378 0.83468973 0.84384537 0.83850458]
Mean accuracy: 0.8407077060602323
Mean precision: 0.8570678908664023
Mean recall: 0.9664944409487124
Mean F1-score: 0.9084934940668973


### TF-IDF weighted embedding features

#### Evaluate TF-IDF weighted embedding representation using 5-fold cross validation

In [47]:
weighted_results = cross_validate(
    model,
    weighted_doc_vectors,
    y,
    cv=cv,
    scoring=scoring_metrics
)

print("Weighted embedding accuracy scores:", weighted_results["test_accuracy"])
print("Mean accuracy:"  , weighted_results["test_accuracy"].mean())
print("Mean precision:" , weighted_results["test_precision"].mean())
print("Mean recall:"    , weighted_results["test_recall"].mean())
print("Mean F1-score:"  , weighted_results["test_f1"].mean())

Weighted embedding accuracy scores: [0.83981693 0.83117213 0.83367243 0.83596134 0.83392675]
Mean accuracy: 0.8349099191725905
Mean precision: 0.849775369417112
Mean recall: 0.9696650002424579
Mean F1-score: 0.9057617244815978


### Q1 result comparison

The average 5-fold cross validation accuracy values are compared to identify which feature representation performs best.

In [48]:
q1_results = pd.DataFrame({
    "Feature Representation": [
        "Count Vector",
        "Unweighted GloVe",
        "TF-IDF Weighted GloVe"
    ],
    "Mean Accuracy": [
        count_results["test_accuracy"].mean(),
        unweighted_results["test_accuracy"].mean(),
        weighted_results["test_accuracy"].mean()
    ],
    "Mean Precision": [
        count_results["test_precision"].mean(),
        unweighted_results["test_precision"].mean(),
        weighted_results["test_precision"].mean()
    ],
    "Mean Recall": [
        count_results["test_recall"].mean(),
        unweighted_results["test_recall"].mean(),
        weighted_results["test_recall"].mean()
    ],
    "Mean F1-Score": [
        count_results["test_f1"].mean(),
        unweighted_results["test_f1"].mean(),
        weighted_results["test_f1"].mean()
    ]
})

q1_results

,Feature Representation,Mean Accuracy,Mean Precision,Mean Recall,Mean F1-Score
0,Count Vector,0.876361,0.904073,0.949649,0.926300
1,Unweighted GloVe,0.840708,0.857068,0.966494,0.908493
2,TF-IDF Weighted GloVe,0.834910,0.849775,0.969665,0.905762


### Q2: Comparing title and review text information

This experiment checks whether adding extra text information improves classification accuracy. Three text inputs are compared: title only, review text only, and title combined with review text.

#### To Prepare title and review text columns

In [49]:
df["Title"]       = df["Title"].fillna("")
df["Review Text"] = df["Review Text"].fillna("")

title_text        = df["Title"]
review_text       = df["Review Text"]
title_review_text = df["Title"] + " " + df["Review Text"]

print("Title examples:")
print(title_text.head())

print("\nReview Text examples:")
print(review_text.head())

print("\nTitle + Review Text examples:")
print(title_review_text.head())

Title examples:
0    Some major design flaws
1           My favorite buy!
2           Flattering shirt
3    Not for the very petite
4       Cagrcoal shimmer fun
Name: Title, dtype: object

Review Text examples:
0    high hopes wanted work initially petite usual ...
1        jumpsuit fun flirty fabulous time compliments
2    shirt due adjustable front tie length leggings...
3    tracy reese dresses petite feet tall brand pre...
4    basket hte person store pick teh pale hte gorg...
Name: Review Text, dtype: object

Title + Review Text examples:
0    Some major design flaws high hopes wanted work...
1    My favorite buy! jumpsuit fun flirty fabulous ...
2    Flattering shirt shirt due adjustable front ti...
3    Not for the very petite tracy reese dresses pe...
4    Cagrcoal shimmer fun basket hte person store p...
dtype: object


#### To Generate count vector features for title only

In [50]:
title_vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"
)

X_title = title_vectorizer.fit_transform(title_text)

print("Title feature matrix shape:", X_title.shape)

Title feature matrix shape: (19662, 3908)


#### To Generate count vector features for title only

In [51]:
title_scores = cross_val_score(model, X_title, y, cv=5, scoring="accuracy")

print("Title only accuracy scores:", title_scores)
print("Mean accuracy:", title_scores.mean())

Title only accuracy scores: [0.88354945 0.88787185 0.88682604 0.87716175 0.88606307]
Mean accuracy: 0.8842944343180624


#### Generate count vector features for review text only

In [52]:
review_vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"
)

X_review = review_vectorizer.fit_transform(review_text)

print("Review text feature matrix shape:", X_review.shape)

Review text feature matrix shape: (19662, 7529)


#### Evaluate review text only features using 5-fold cross validation

In [53]:
review_scores = cross_val_score(model, X_review, y, cv=5, scoring="accuracy")

print("Review text only accuracy scores:", review_scores)
print("Mean accuracy:", review_scores.mean())

Review text only accuracy scores: [0.87465039 0.87871854 0.88479145 0.87131231 0.86876907]
Mean accuracy: 0.8756483535641113


#### Generate count vector features for title + review text

In [54]:
combined_vectorizer = CountVectorizer(
    lowercase=True,
    token_pattern=r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"
)

X_combined = combined_vectorizer.fit_transform(title_review_text)

print("Combined feature matrix shape:", X_combined.shape)

Combined feature matrix shape: (19662, 8848)


#### Evaluate title + review text features using 5-fold cross validation

In [55]:
combined_scores = cross_val_score(model, X_combined, y, cv=5, scoring="accuracy")

print("Title + review text accuracy scores:", combined_scores)
print("Mean accuracy:", combined_scores.mean())

Title + review text accuracy scores: [0.90338164 0.90211035 0.90005086 0.89242116 0.89140387]
Mean accuracy: 0.8978735761957861


### Q2 result comparison

In [56]:
q2_results = pd.DataFrame({
    "Text Information Used": [
        "Title only",
        "Review Text only",
        "Title + Review Text"
    ],
    "Mean Accuracy": [
        title_scores.mean(),
        review_scores.mean(),
        combined_scores.mean()
    ]
})

q2_results

,Text Information Used,Mean Accuracy
0,Title only,0.884294
1,Review Text only,0.875648
2,Title + Review Text,0.897874


## Summary
In Task 3, Logistic Regression was used as the selected classification model.
- For Q1, three feature representations from Task 2 were compared using 5-fold cross validation: count vectors, unweighted GloVe vectors, and TF-IDF weighted GloVe vectors.
- For Q2, three text input settings were compared: title only, review text only, and title combined with review text. The results showed that using both title and review text gave the highest accuracy, and the TF-IDF weighted GloVe representation performed best among the three feature representations.